## Centerline

In [17]:
import numpy
import sys
from scipy.ndimage import distance_transform_edt
from skimage.feature import peak_local_max
from scipy.spatial import cKDTree
import vtk
from sklearn.decomposition import PCA
from scipy.interpolate import UnivariateSpline

sys.path.insert(0, "../src")  # add Folder_2 path to search list

from meshing_functions import export_centerline, export_mask, set_the_offset

base = "/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/"

offset = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/meshes/offset.txt")
pixel_spacing = numpy.loadtxt("/Users/skardova/Documents/MRI_data/Brain/Comprehensive ultrahigh resolution/ses-4712/pixel_spacing.txt")

out_folder = "centerlines/"

mask_filename_base = "Segmentation_1_filtered_surf_" # + id=1.npy



## 1.  Load mask

In [ ]:
id = 5
nbins = 15
trim = 0.05
smoothness = 2
spline_order = 3

mask = numpy.load(base + mask_filename_base + "id_"+str(id)+".npy")
# export_mask(mask, base + mask_filename_base + "id="+str(id)+".vti")

points = numpy.argwhere(mask)  # (z,y,x)
# points = points[:, [2,1,0]]  # convert to (x,y,z)




## 2. Compute principal axis (using PCA) -- Parametrize

In [19]:
pca = PCA(n_components=3)
pca.fit(points)

axis = pca.components_[0]      # main tube direction
origin = pca.mean_
print("axis = ", axis)

t = (points - origin) @ axis    # Parameterize coordinates along the axis


order = numpy.argsort(t)        
t_sorted = t[order]
P = points[order]


axis =  [ 0.37145715  0.85769254 -0.35550399]


## 3. Remove ends


In [20]:
tmin = t.min()
tmax = t.max()

t_lo = tmin + trim * (tmax - tmin)
t_hi = tmax - trim * (tmax - tmin)

core_mask = (t >= t_lo) & (t <= t_hi)

points_core = points[core_mask]
t_core = t[core_mask]


## 4. Compute cross-section centers 

 - Divide the tube into bins along the axis 
 - Compute the centroid of points in each bin

In [21]:
bins = numpy.linspace(t_lo, t_hi, nbins)

centers = []
t_centers = []

for i in range(nbins-1):

    # points belonging to the current slice along the tube
    m = (t_core >= bins[i]) & (t_core < bins[i+1])

    if numpy.sum(m) > 0:
        centers.append(points_core[m].mean(axis=0))  # centroid of slice
        t_centers.append(t_core[m].mean())           # average axial position

centers = numpy.array(centers)
t_centers = numpy.array(t_centers)

## 5. Fit parametric centerline
- Independent splines for x(t), y(t), z(t)
- The smoothing factor 's' controls how closely the spline follows the centroid points (higher s = smoother curve)

In [22]:
sx = UnivariateSpline(t_centers, centers[:,0], k = spline_order, s = smoothness)
sy = UnivariateSpline(t_centers, centers[:,1], k = spline_order, s = smoothness)
sz = UnivariateSpline(t_centers, centers[:,2], k = spline_order, s = smoothness)

In [23]:
tt = numpy.linspace(t_sorted.min(), t_sorted.max(), 200)

centerline = numpy.vstack([
    sx(tt),
    sy(tt),
    sz(tt)
]).T


export_centerline(centerline, pixel_spacing[0], base + out_folder + mask_filename_base + "_centerline_id_"+str(id)+".vtp", offset)


In [24]:
tck_x = sx._eval_args
tck_y = sy._eval_args
tck_z = sz._eval_args

numpy.savez(
    base + out_folder + "centerline_id_"+str(id)+"_parametric.npz",
    tx=tck_x[0], cx=tck_x[1], kx=tck_x[2],
    ty=tck_y[0], cy=tck_y[1], ky=tck_y[2],
    tz=tck_z[0], cz=tck_z[1], kz=tck_z[2]
)